# Notebook 03 — Feature Engineering


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "online_food_delivery_analysis_cleaned.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "online_food_delivery_analysis_featured.csv"

print("Project root:", PROJECT_ROOT.resolve())
print("Input path:", INPUT_PATH.resolve())
print("Input file exists:", INPUT_PATH.exists())

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        "Cleaned dataset not found. Run Notebook 02 first and confirm that "
        "data/processed/online_food_delivery_analysis_cleaned.csv exists."
    )

df = pd.read_csv(INPUT_PATH)
print("Input shape:", df.shape)
display(df.head())


Project root: D:\online food delivery analysis final
Input path: D:\online food delivery analysis final\data\processed\online_food_delivery_analysis_cleaned.csv
Input file exists: True
Input shape: (100000, 25)


,Order_ID,Customer_ID,Customer_Age,Customer_Gender,City,Area,Restaurant_ID,Restaurant_Name,Cuisine_Type,Order_Date,...,Final_Amount,Payment_Mode,Order_Status,Cancellation_Reason,Delivery_Partner_ID,Delivery_Rating,Restaurant_Rating,Order_Day,Peak_Hour,Profit_Margin
0,ORD000001,CUST03987,24.0,Male,Chennai,OMR,RES1298,Restaurant_1298,Fast Food,2024-06-05,...,1801.20,COD,Delivered,Not Applicable,DP00866,4.4,3.2,Weekday,False,0.029
1,ORD000002,CUST01917,45.0,Female,Hyderabad,NaN,RES1191,Restaurant_1191,Chinese,2024-05-11,...,469.78,Credit Card,Delivered,Not Applicable,DP04095,4.6,4.3,Weekend,False,0.214
2,ORD000003,CUST13338,57.0,Male,Bengaluru,Indiranagar,RES0681,Restaurant_0681,Indian,2024-02-02,...,468.45,UPI,Delivered,Not Applicable,DP01853,4.6,3.5,Weekday,False,0.204
3,ORD000004,CUST09787,66.0,Female,Chennai,Velachery,RES0453,Restaurant_0453,Indian,2024-03-27,...,681.18,UPI,Delivered,Not Applicable,DP04885,5.0,4.3,Weekday,True,0.293
4,ORD000005,CUST15793,43.0,Female,Chennai,Anna Nagar,RES0268,Restaurant_0268,Indian,2024-07-11,...,471.87,UPI,Preparing,Not Applicable,DP04754,NaN,4.7,Weekday,True,NaN


## 1. Check required columns


In [2]:
required = [
    "Order_Date", "Order_Time", "Order_Status", "Customer_Age",
    "Delivery_Time_Min", "Distance_km", "Order_Value",
    "Discount_Applied", "Final_Amount", "Profit_Margin",
    "Order_Day", "Peak_Hour"
]
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f"Required columns missing from cleaned dataset: {missing}")

original_columns = df.columns.tolist()
original_row_count = len(df)
print("All required columns are present.")
print("Original columns:", len(original_columns))
print("Rows:", original_row_count)


All required columns are present.
Original columns: 25
Rows: 100000


## 2. Create date and time features


In [3]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df["Order_Time_Parsed"] = pd.to_datetime(
    df["Order_Time"].astype("string"),
    format="%H:%M:%S",
    errors="coerce"
)
# Some CSVs may store times as HH:MM, so retry those values that did not parse.
time_missing = df["Order_Time_Parsed"].isna() & df["Order_Time"].notna()
if time_missing.any():
    retry = pd.to_datetime(
        df.loc[time_missing, "Order_Time"].astype("string"),
        format="%H:%M",
        errors="coerce"
    )
    df.loc[time_missing, "Order_Time_Parsed"] = retry

df["Order_Year"] = df["Order_Date"].dt.year.astype("Int64")
df["Order_Month"] = df["Order_Date"].dt.month.astype("Int64")
df["Order_Month_Name"] = df["Order_Date"].dt.month_name()
df["Order_Weekday_Name"] = df["Order_Date"].dt.day_name()
df["Order_Hour"] = df["Order_Time_Parsed"].dt.hour.astype("Int64")
df["Is_Weekend"] = df["Order_Date"].dt.dayofweek.isin([5, 6]).where(df["Order_Date"].notna(), pd.NA).astype("boolean")

print("Date/time features created.")
display(df[["Order_Date", "Order_Time", "Order_Year", "Order_Month", "Order_Month_Name",
            "Order_Weekday_Name", "Order_Hour", "Is_Weekend"]].head())
print("Missing parsed order dates:", int(df["Order_Date"].isna().sum()))
print("Missing parsed order times:", int(df["Order_Time_Parsed"].isna().sum()))


Date/time features created.


,Order_Date,Order_Time,Order_Year,Order_Month,Order_Month_Name,Order_Weekday_Name,Order_Hour,Is_Weekend
0,2024-06-05,14:11:00,2024,6,June,Wednesday,14,False
1,2024-05-11,04:09:00,2024,5,May,Saturday,4,True
2,2024-02-02,01:23:00,2024,2,February,Friday,1,False
3,2024-03-27,19:06:00,2024,3,March,Wednesday,19,False
4,2024-07-11,07:47:00,2024,7,July,Thursday,7,False


Missing parsed order dates: 0
Missing parsed order times: 0


## 3. Create peak-hour features


In [4]:
def peak_period(hour):
    if pd.isna(hour):
        return pd.NA
    hour = int(hour)
    if 7 <= hour <= 9:
        return "Morning Peak"
    if 12 <= hour <= 13:
        return "Lunch Peak"
    if 18 <= hour <= 21:
        return "Evening Peak"
    return "Off-Peak"

df["Peak_Period"] = df["Order_Hour"].map(peak_period).astype("string")
df["Peak_Hour_Recalculated"] = df["Order_Hour"].isin([7, 8, 9, 12, 13, 18, 19, 20, 21]).where(
    df["Order_Hour"].notna(), pd.NA
).astype("boolean")

if "Peak_Hour" in df.columns:
    existing_peak = df["Peak_Hour"].astype("string").str.lower().map({"true": True, "false": False})
    comparable = existing_peak.notna() & df["Peak_Hour_Recalculated"].notna()
    mismatch_count = int((existing_peak[comparable].astype(bool) != df.loc[comparable, "Peak_Hour_Recalculated"].astype(bool)).sum())
    print("Peak_Hour mismatches against proposed rule:", mismatch_count)

display(df[["Order_Time", "Order_Hour", "Peak_Hour", "Peak_Period", "Peak_Hour_Recalculated"]].head())


Peak_Hour mismatches against proposed rule: 0


,Order_Time,Order_Hour,Peak_Hour,Peak_Period,Peak_Hour_Recalculated
0,14:11:00,14,False,Off-Peak,False
1,04:09:00,4,False,Off-Peak,False
2,01:23:00,1,False,Off-Peak,False
3,19:06:00,19,True,Evening Peak,True
4,07:47:00,7,True,Morning Peak,True


## 4. Create customer age groups

In [5]:
age = pd.to_numeric(df["Customer_Age"], errors="coerce")
age_bins = [0, 17, 25, 35, 45, 60, np.inf]
age_labels = ["17 or younger", "18–25", "26–35", "36–45", "46–60", "Over 60"]

df["Customer_Age_Group"] = pd.cut(
    age,
    bins=age_bins,
    labels=age_labels,
    right=True,
    include_lowest=True
).astype("string")

# Keep invalid/implausible ages unclassified for review.
invalid_age = age.notna() & ((age < 0) | (age > 100))
df.loc[invalid_age, "Customer_Age_Group"] = pd.NA

print("Age groups:")
display(df["Customer_Age_Group"].value_counts(dropna=False).rename_axis("Customer_Age_Group").to_frame("Orders"))


Age groups:


,Orders
Customer_Age_Group,
46–60,27937
26–35,18683
36–45,18499
Over 60,18462
18–25,14943
<NA>,1476


## 5. Create delivery performance categories

In [6]:
delivery_time = pd.to_numeric(df["Delivery_Time_Min"], errors="coerce")
is_delivered = df["Order_Status"].astype("string").str.strip().str.casefold().eq("delivered")

conditions = [
    is_delivered & delivery_time.notna() & (delivery_time <= 30),
    is_delivered & delivery_time.notna() & (delivery_time > 30) & (delivery_time <= 60),
    is_delivered & delivery_time.notna() & (delivery_time > 60)
]
choices = ["Fast", "Standard", "Slow"]

df["Delivery_Performance"] = pd.Series(
    np.select(conditions, choices, default="Unclassified"),
    index=df.index,
    dtype="string"
)
df.loc[~is_delivered | delivery_time.isna(), "Delivery_Performance"] = pd.NA

print("Delivery performance categories:")
display(df["Delivery_Performance"].value_counts(dropna=False).rename_axis("Delivery_Performance").to_frame("Orders"))


Delivery performance categories:


,Orders
Delivery_Performance,
Standard,46759
Slow,40700
<NA>,10712
Fast,1829


## 6. Create financial features

In [7]:
order_value = pd.to_numeric(df["Order_Value"], errors="coerce")
discount = pd.to_numeric(df["Discount_Applied"], errors="coerce")
final_amount = pd.to_numeric(df["Final_Amount"], errors="coerce")
profit_margin = pd.to_numeric(df["Profit_Margin"], errors="coerce")

df["Discount_Percentage"] = np.where(
    order_value.notna() & discount.notna() & (order_value > 0),
    (discount / order_value) * 100,
    np.nan
)

df["Profit_Margin_Percentage"] = profit_margin * 100

df["Order_Value_Band"] = pd.cut(
    order_value,
    bins=[-np.inf, 500, 1000, 2000, np.inf],
    labels=["Up to 500", "501–1,000", "1,001–2,000", "Above 2,000"]
).astype("string")

print("Financial features created.")
display(df[["Order_Value", "Discount_Applied", "Discount_Percentage",
            "Profit_Margin", "Profit_Margin_Percentage", "Order_Value_Band"]].head())


Financial features created.


,Order_Value,Discount_Applied,Discount_Percentage,Profit_Margin,Profit_Margin_Percentage,Order_Value_Band
0,1901.20,100.0,5.259836,0.029,2.9,"1,001–2,000"
1,489.78,20.0,4.083466,0.214,21.4,Up to 500
2,468.45,0.0,0.000000,0.204,20.4,Up to 500
3,731.18,50.0,6.838261,0.293,29.3,"501–1,000"
4,571.87,100.0,17.486492,NaN,NaN,"501–1,000"


## 7. Create cancellation and delivery indicators

In [8]:
status = df["Order_Status"].astype("string").str.strip().str.casefold()
df["Is_Cancelled"] = status.eq("cancelled").astype("int8")
df["Is_Delivered"] = status.eq("delivered").astype("int8")
df["Is_Non_Delivered"] = (~status.eq("delivered")).astype("int8")

print("Order status indicator counts:")
display(df[["Is_Cancelled", "Is_Delivered", "Is_Non_Delivered"]].sum().rename("Count").to_frame())


Order status indicator counts:


,Count
Is_Cancelled,6143
Is_Delivered,89288
Is_Non_Delivered,10712


## 8. Validate the engineered features

In [9]:
validation = {
    "Rows preserved": len(df) == original_row_count,
    "All original columns retained": all(column in df.columns for column in original_columns),
    "New feature columns added": len(df.columns) - len(original_columns),
    "Delivered flag matches status": bool((df["Is_Delivered"] == df["Order_Status"].astype("string").str.strip().str.casefold().eq("delivered").astype("int8")).all()),
    "Cancelled flag matches status": bool((df["Is_Cancelled"] == df["Order_Status"].astype("string").str.strip().str.casefold().eq("cancelled").astype("int8")).all()),
    "Profit percentage matches ratio × 100": bool(np.allclose(
        df["Profit_Margin_Percentage"].to_numpy(dtype=float),
        (pd.to_numeric(df["Profit_Margin"], errors="coerce") * 100).to_numpy(dtype=float),
        equal_nan=True
    ))
}
display(pd.Series(validation, name="Passed").to_frame())

if not validation["Rows preserved"] or not validation["All original columns retained"]:
    raise AssertionError("Feature engineering changed row count or removed source columns.")


,Passed
Rows preserved,True
All original columns retained,True
New feature columns added,17
Delivered flag matches status,True
Cancelled flag matches status,True
Profit percentage matches ratio × 100,True


## 9. Save the feature-engineered dataset

In [10]:
df.to_csv(OUTPUT_PATH, index=False)

print("Feature-engineered dataset saved to:", OUTPUT_PATH.resolve())
print("Output exists:", OUTPUT_PATH.exists())
print("Output shape:", df.shape)

# Reload to verify the saved file.
df_saved = pd.read_csv(OUTPUT_PATH)
print("Reloaded output shape:", df_saved.shape)
assert df_saved.shape == df.shape, "Saved dataset shape does not match in-memory DataFrame."
print("Saved output validation: passed.")


Feature-engineered dataset saved to: D:\online food delivery analysis final\data\processed\online_food_delivery_analysis_featured.csv
Output exists: True
Output shape: (100000, 42)
Reloaded output shape: (100000, 42)
Saved output validation: passed.
